# 03 · Policy Documents — RAG Source Material
4 real policy docs (fraud handling, KYC/AML, loan processing, refund/dispute) — the retrieval corpus for the RAG layer.

In [1]:
import sys
sys.path.append('../src')
from pathlib import Path
from data_loader import list_policy_docs
docs = list_policy_docs()
for d in docs:
    words = len(d.read_text().split())
    print(f'{d.name}: {words} words, {d.stat().st_size/1024:.1f} KB')

fraud_handling_policy.txt: 468 words, 3.2 KB
kyc_policy.txt: 378 words, 2.5 KB
loan_processing_policy.txt: 466 words, 2.9 KB
refund_dispute_policy.txt: 292 words, 2.0 KB


## 1. Chunking check
`src/rag_pipeline.py` chunks each doc into ~90-word pieces (15-word overlap). At these lengths, each policy doc becomes roughly 5-7 chunks — small enough that chunking mostly preserves whole sections rather than fragmenting mid-thought.

In [2]:
import sys; sys.path.append('../src')
from rag_pipeline import chunk_text
for d in docs:
    n_chunks = len(chunk_text(d.read_text()))
    print(f'{d.name}: {n_chunks} chunks')

fraud_handling_policy.txt: 7 chunks
kyc_policy.txt: 6 chunks
loan_processing_policy.txt: 7 chunks
refund_dispute_policy.txt: 4 chunks


## 2. Category coverage check (vs. ticket categories from notebook 01)
Every ticket category (Fraud, Loan, KYC, Account Access) needs a policy doc. 3 of 4 map directly by name; Account Access tickets don't have a dedicated policy doc — they're likely meant to draw on the KYC/fraud docs (password reset, IPIN reset) or there's a genuine coverage gap worth flagging to whoever owns the policy library.

In [3]:
doc_categories = {
    'fraud_handling_policy': 'Fraud',
    'kyc_policy': 'KYC',
    'loan_processing_policy': 'Loan',
    'refund_dispute_policy': 'Fraud/Refund (overlaps Fraud)',
}
expected = {'Fraud', 'Loan', 'KYC', 'Account Access'}
covered = {'Fraud', 'Loan', 'KYC'}
print('Covered:', covered)
print('Missing dedicated policy doc:', expected - covered)

Covered: {'Fraud', 'Loan', 'KYC'}
Missing dedicated policy doc: {'Account Access'}


## 3. Retrieval quality against qa_pairs.json ground truth
20 curated Q&A pairs with a `policy_ref` field naming the correct source policy — this is real ground truth for evaluating retrieval, not a proxy metric.

**Finding during development:** a single shared TF-IDF index (policies + all 200 ticket chunks together) let near-duplicate tickets crowd out policy chunks — policy hit rate was only 15%. Splitting into two separate indices (policies, past cases) and merging top results from each fixed it: hit rate jumped to 75%. See `src/rag_pipeline.py` for the fix and rationale.

In [4]:
from rag_pipeline import evaluate_retrieval, build_index
build_index()
evaluate_retrieval()

Indexed 24 policy chunks + 200 ticket chunks -> /home/claude/banking_rag_analysis/data/processed/retrieval_index.joblib


{'n_questions': 20,
 'policy_hit_rate': 0.75,
 'avg_policy_similarity_when_hit': 0.204}

## 4. Takeaways
- 4 policy docs, cleanly chunkable (~90-word chunks, no obvious mid-sentence breaks).
- Account Access has no dedicated policy doc — flag this to the policy team, or route those queries to KYC/fraud docs explicitly.
- Retrieval reaches a 75% policy-hit-rate against the qa_pairs ground truth after fixing the dual-index issue; remaining misses are largely a lexical (TF-IDF) vocabulary gap between conversational query phrasing and formal policy language — see the RAG pipeline notes on upgrading to dense embeddings.